<a href="https://colab.research.google.com/github/Arzuyasar/amazon-customer-review/blob/main/Faz3_NLP_Pipeline%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q vaderSentiment transformers sumy torch tqdm

import pandas as pd
import numpy as np
import os, gc, time
from tqdm.auto import tqdm
from IPython.display import display

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/'
print("Hazır")

In [ ]:
df = pd.read_parquet(DRIVE_PATH + 'faz2_temiz_veri.parquet')
print(f"{len(df):,} satır yüklendi")

ORNEKLEM = 200_000
df_nlp   = df.head(ORNEKLEM).copy()
print(f"   Çalışılacak satır: {len(df_nlp):,} (geliştirme modu)")
print("   Final için ORNEKLEM değişkenini kaldır veya büyüt")

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def vader_skor_hesapla(text: str) -> float:

    return analyzer.polarity_scores(str(text))['compound']

print("VADER skoru hesaplanıyor...")
df_nlp['vader_compound'] = df_nlp['review_body'].apply(vader_skor_hesapla)

print(f"Tamamlandı")
print("\nVADER skoru dağılımı:")
print(df_nlp['vader_compound'].describe().round(3).to_string())

print("\nÖrnek yorumlar:")
print("─" * 70)
ornekler = df_nlp[['review_body','star_rating','vader_compound']].sample(5)
for _, row in ornekler.iterrows():
    print(f"Yorum: {str(row['review_body'])[:70]}...")
    print(f" {row['star_rating']}  VADER: {row['vader_compound']:+.3f}\n")

In [ ]:
def problem_tespit(star_rating, vader_compound):
    if vader_compound < -0.3:
        return 1
    if vader_compound > 0.2:
        return 0
    yildiz_sinyal = int(star_rating <= 3)
    vader_sinyal  = int(vader_compound < -0.05)
    skor          = yildiz_sinyal * 0.4 + vader_sinyal * 0.6
    return int(skor >= 0.5)

df_nlp['problem_var'] = df_nlp.apply(
    lambda r: problem_tespit(r['star_rating'], r['vader_compound']), axis=1
)

print(f"Problem tespit sonuçları:")
print(f"  Problemli yorum : {df_nlp['problem_var'].sum():,}  (%{df_nlp['problem_var'].mean()*100:.1f})")
print(f"  Sorunsuz yorum  : {(1-df_nlp['problem_var']).sum():,}  (%{(1-df_nlp['problem_var'].mean())*100:.1f})")

In [ ]:
from transformers import pipeline as hf_pipeline

print("Model yükleniyor (ilk seferinde 1-2 dk sürer)...")
siniflandirici = hf_pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=0,
)
print("Model hazır")

ETIKETLER = [
    'hardware defect or malfunction',
    'software bug or app issue',
    'shipping and delivery problem',
    'customer service complaint',
    'product design or usability issue',
    'positive review no problem',
]

TURKCE = {
    'hardware defect or malfunction':     'Teknik Destek',
    'software bug or app issue':          'Yazılım Ekibi',
    'shipping and delivery problem':      'Lojistik',
    'customer service complaint':         'Müşteri Hizmetleri',
    'product design or usability issue':  'Ürün Yönetimi',
    'positive review no problem':         'Arşiv (Olumlu)',
}

def departman_bul(metin: str) -> dict:

    sonuc    = siniflandirici(str(metin)[:512], ETIKETLER, multi_label=False)
    en_iyi   = sonuc['labels'][0]
    return {
        'departman_en': en_iyi,
        'departman_tr': TURKCE[en_iyi],
        'guven_skoru':  round(sonuc['scores'][0], 3),
    }

print("\nTest sonuçları:")
print("─" * 70)
test_metinler = [
    "The screen cracked after one week. Hardware quality is terrible.",
    "App crashes every time I open it. Please fix the software.",
    "Package arrived completely crushed. Box was destroyed during shipping.",
    "Customer service never replied to my 5 emails. Unacceptable.",
    "Absolutely love this product! Best purchase I made this year.",
]
for metin in test_metinler:
    sonuc = departman_bul(metin)
    print(f"Yorum    : {metin[:60]}...")
    print(f"Departman: {sonuc['departman_tr']}  (güven: {sonuc['guven_skoru']:.2f})\n")

In [ ]:
!pip install -q transformers torch tqdm

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

PROBLEM_KELIMELERI_DEPT = {
    'Teknik Destek': [
        'broken', 'defect', 'malfunction', 'not working', 'stopped working',
        'hardware', 'damage', 'cracked', 'dead', 'faulty', 'repair',
        'doesnt work', "doesn't work", 'stopped', 'fell apart',
        'freeze', 'freezing', 'stuck', 'unresponsive', 'corrupt',
        'overheating', 'battery dead', 'screen cracked', 'no power',
        'wont turn on', 'no sound', 'physical damage',
        'worst hardware', 'terrible hardware', 'awful hardware',
    ],
    'Yazılım Ekibi': [
        'software', 'app', 'crash', 'crashing', 'bug', 'update', 'glitch',
        'error', 'install', 'compatible', 'firmware', 'driver', 'reboot',
        'bluetooth', 'wifi', 'sync', 'patch', 'reinstall', 'lag', 'slow',
        'not loading', 'wont open', 'keeps closing', 'login issue',
        'sign in', 'timeout', 'authentication', 'server', 'connection issue',
        'worst app', 'terrible app', 'awful app', 'bad app',
        'worst software', 'terrible software', 'app not working',
    ],
    'Lojistik': [
        'delivery', 'shipping', 'package', 'arrived', 'shipment',
        'damaged box', 'missing', 'lost', 'never received',
        'wrong item', 'not delivered', 'still waiting', 'stolen',
        'opened box', 'repackaged', 'weeks late', 'days late',
        'late delivery', 'delayed', 'delay', 'customs',
        'package damaged', 'worst delivery', 'terrible delivery',
        'awful delivery', 'bad delivery', 'poor delivery', 'slow delivery',
        'worst shipping', 'terrible shipping', 'never arrived',
        'wrong address', 'package lost', 'return shipping',
    ],
    'Müşteri Hizmetleri': [
        'customer service', 'support', 'refund', 'warranty', 'response',
        'replied', 'contact', 'ignored', 'rude', 'unhelpful',
        'representative', 'call center', 'exchange', 'complaint', 'no reply',
        'not responding', 'escalate', 'manager', 'supervisor', 'promised',
        'lied', 'scam', 'fraud', 'dispute', 'return policy', 'never called',
        'worst service', 'terrible service', 'awful service', 'bad service',
        'poor service', 'horrible service', 'worst support', 'useless support',
    ],
    'Ürün Yönetimi': [
        'design', 'quality', 'cheap', 'material', 'size', 'color',
        'uncomfortable', 'usability', 'confusing', 'misleading', 'overpriced',
        'build quality', 'poorly made', 'flimsy', 'smell', 'cheap plastic',
        'breaks easily', 'not as described', 'different from photo',
        'not worth', 'false advertising', 'wrong color', 'wrong size',
        'terrible product', 'awful product', 'horrible product', 'worst product',
        'poor quality', 'bad quality', 'low quality', 'cheap quality',
        'terrible quality', 'awful quality', 'not worth the money',
        'waste of money', 'disappointing product',
    ],
}

GENEL_PROBLEM = [
    'terrible', 'awful', 'horrible', 'worst', 'useless', 'disappointed',
    'waste', 'regret', 'never again', 'do not buy', 'avoid', 'zero stars',
    'one star', 'not happy', 'very bad', 'really bad', 'extremely bad',
    'issue', 'problem', 'fail', 'failed', 'failure', 'cannot', 'unable',
    'keeps', 'still not fixed', 'still not working',
]

POZITIF_IFADELER = {
    'Lojistik': [
        'fast delivery', 'quick delivery', 'great delivery', 'perfect delivery',
        'amazing delivery', 'excellent delivery', 'speedy delivery',
        'fast shipping', 'quick shipping', 'free shipping', 'on time',
        'arrived on time', 'arrived quickly', 'delivered fast',
        'great packaging', 'perfect packaging', 'well packaged',
    ],
    'Müşteri Hizmetleri': [
        'great service', 'excellent service', 'amazing service',
        'helpful service', 'great support', 'excellent support',
        'amazing support', 'helpful support', 'great customer service',
        'excellent customer service', 'fast response', 'quick response',
        'very helpful', 'very responsive',
    ],
    'Yazılım Ekibi': [
        'great app', 'excellent app', 'amazing app', 'works perfectly',
        'works great', 'no bugs', 'runs smoothly', 'great software',
        'easy to use', 'user friendly',
    ],
    'Teknik Destek': [
        'works perfectly', 'works great', 'no issues', 'great hardware',
        'excellent build', 'solid build', 'excellent quality',
        'perfect condition', 'no problems',
    ],
}

KURALLAR = PROBLEM_KELIMELERI_DEPT

def metin_problemi_var_mi(metin: str) -> bool:
    metin_lower = str(metin).lower()
    if any(k in metin_lower for k in GENEL_PROBLEM):
        return True
    for kelimeler in PROBLEM_KELIMELERI_DEPT.values():
        if any(k in metin_lower for k in kelimeler):
            return True
    return False

def problem_tespit(star_rating, vader_compound, review_body="") -> int:
    teknik = metin_problemi_var_mi(review_body)

    if vader_compound > 0.5:
        return 0
    if vader_compound < -0.3:
        return 1
    if teknik and vader_compound > 0.1:
        return 0
    if teknik and vader_compound <= 0.1:
        return 1

    yildiz_sinyal = int(star_rating <= 3)
    vader_sinyal  = int(vader_compound < -0.05)
    skor          = yildiz_sinyal * 0.4 + vader_sinyal * 0.6
    return int(skor >= 0.5)

def kural_ile_departman(metin: str) -> dict:
    metin_lower = str(metin).lower()
    skorlar = {}

    for dept, kelimeler in PROBLEM_KELIMELERI_DEPT.items():
        eslesme = sum(1 for k in kelimeler if k in metin_lower)
        genel   = sum(1 for k in GENEL_PROBLEM if k in metin_lower)
        skorlar[dept] = eslesme + genel * 0.3

    for dept, pozitif_kelimeler in POZITIF_IFADELER.items():
        pozitif_var = any(k in metin_lower for k in pozitif_kelimeler)
        negatif_var = any(k in metin_lower for k in PROBLEM_KELIMELERI_DEPT.get(dept, []))
        if pozitif_var and not negatif_var:
            skorlar[dept] = 0

    en_iyi   = max(skorlar, key=skorlar.get)
    max_skor = skorlar[en_iyi]

    if max_skor == 0:
        return {'departman_tr': 'Teknik Destek', 'guven_skoru': 0.25}

    guven = round(min(0.45 + max_skor * 0.08, 0.90), 2)
    return {'departman_tr': en_iyi, 'guven_skoru': guven}

print("Problem tespiti yapılıyor...")
df_nlp['problem_var'] = df_nlp.apply(
    lambda r: problem_tespit(
        r['star_rating'],
        r['vader_compound'],
        r['review_body']
    ), axis=1
)
print(f"Problemli: {df_nlp['problem_var'].sum():,}  (%{df_nlp['problem_var'].mean()*100:.1f})")

print("\nKural tabanlı departman sınıflandırması başlıyor...")
problemli_idx = df_nlp[df_nlp['problem_var'] == 1].index
print(f"Toplam problemli yorum: {len(problemli_idx):,}")

for sutun in ['departman_tr', 'guven_skoru']:
    if sutun in df_nlp.columns:
        df_nlp = df_nlp.drop(columns=[sutun])

tqdm.pandas(desc="Departman atanıyor")
sonuclar = df_nlp.loc[problemli_idx, 'review_body'].progress_apply(kural_ile_departman)
sonuc_df = pd.DataFrame(sonuclar.tolist(), index=problemli_idx)
df_nlp   = df_nlp.join(sonuc_df)

df_nlp.loc[df_nlp['problem_var'] == 0, 'departman_tr'] = 'Arşiv (Olumlu)'
df_nlp.loc[df_nlp['problem_var'] == 0, 'guven_skoru']  = 1.0

print("\n✅ Tamamlandı")
print("\nDepartman dağılımı:")
print(df_nlp['departman_tr'].value_counts().to_string())
print("\nGüven skoru dağılımı:")
print(df_nlp.loc[problemli_idx, 'guven_skoru'].describe().round(2).to_string())

print("\n── Test Senaryoları ─────────────────────────────────────────────────")
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

test_yorumlar = [
    ("The app keeps crashing every time I try to log in.", 1),
    ("Package arrived 3 weeks late and was completely damaged.", 2),
    ("Customer service ignored all my emails for weeks.", 1),
    ("Cheap plastic, breaks easily, not as described.", 3),
    ("Screen cracked after 1 day, faulty hardware.", 1),
    ("Amazing product! Works perfectly, fast delivery.", 5),
    ("Terrible product, worst purchase ever.", 4),
    ("Worst delivery I have ever seen. Package arrived crushed.", 5),
    ("Great product, quick delivery, very happy!", 5),
    ("Delivery was delayed by 3 weeks, terrible.", 2),
    ("Fast shipping, arrived on time, perfect packaging.", 5),
    ("Worst customer service ever, ignored all my emails.", 1),
    ("Great customer service, very helpful and responsive.", 5),
]

print(f"\n{'Sonuç':<15} {'Departman':<24} {'Yorum'}")
print("-" * 80)
for yorum, yildiz in test_yorumlar:
    vader  = analyzer.polarity_scores(yorum)["compound"]
    sonuc  = problem_tespit(yildiz, vader, yorum)
    dept   = kural_ile_departman(yorum) if sonuc == 1 else {"departman_tr": "Arşiv (Olumlu)"}
    etiket = "⚠️ Problemli" if sonuc == 1 else "✅ Sorunsuz"
    print(f"{etiket:<15} {dept['departman_tr']:<24} {yorum[:50]}")

In [ ]:
GUVEN_ESIK  = 0.45
MAX_ZEROSHT = 500

dusuk_guven_idx = df_nlp[
    (df_nlp['problem_var'] == 1) &
    (df_nlp['guven_skoru'] < GUVEN_ESIK)
].index

print(f"Düşük güvenli yorum sayısı : {len(dusuk_guven_idx):,}")
print(f"Zero-shot'a gidecek        : {min(len(dusuk_guven_idx), MAX_ZEROSHT):,}")
print(f"Toplam yorum sayısı       : {len(df_nlp):,}")
from transformers import pipeline as hf_pipeline

print("\nZero-shot modeli yükleniyor...")
print("(İlk seferinde ~800 MB indirir, sonraki çalışmalarda önbellekten gelir)")

siniflandirici = hf_pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=0,
)
print("Model hazır")

ETIKETLER = [
    'hardware defect or malfunction',
    'software bug or app issue',
    'shipping and delivery problem',
    'customer service complaint',
    'product design or usability issue',
]
TURKCE = {
    'hardware defect or malfunction':    'Teknik Destek',
    'software bug or app issue':         'Yazılım Ekibi',
    'shipping and delivery problem':     'Lojistik',
    'customer service complaint':        'Müşteri Hizmetleri',
    'product design or usability issue': 'Ürün Yönetimi',
}

def zero_shot_departman(metin: str) -> dict:
    try:
        sonuc  = siniflandirici(str(metin)[:512], ETIKETLER, multi_label=False)
        en_iyi = sonuc['labels'][0]
        return {
            'departman_tr': TURKCE[en_iyi],
            'guven_skoru':  round(sonuc['scores'][0], 3),
        }
    except Exception:
        return {'departman_tr': 'Teknik Destek', 'guven_skoru': 0.3}

In [ ]:
hedef_idx = dusuk_guven_idx[:MAX_ZEROSHT]
print(f"\nZero-shot sınıflandırma başlıyor ({len(hedef_idx):,} yorum)...")

zero_sonuclar = []
BATCH = 16

for i in tqdm(range(0, len(hedef_idx), BATCH), desc="Zero-shot"):
    batch_idx = hedef_idx[i:i+BATCH]
    for idx in batch_idx:
        metin = df_nlp.loc[idx, 'review_body']
        zero_sonuclar.append(zero_shot_departman(metin))

zero_df = pd.DataFrame(zero_sonuclar, index=hedef_idx)
df_nlp.loc[hedef_idx, 'departman_tr'] = zero_df['departman_tr'].values
df_nlp.loc[hedef_idx, 'guven_skoru']  = zero_df['guven_skoru'].values

print("\nZero-shot doğrulama tamamlandı")

In [ ]:
print("── Sınıflandırma Tamamlandı ─────────────────────────────")
print(f"Toplam yorum        : {len(df_nlp):,}")
print(f"Problemli           : {df_nlp['problem_var'].sum():,}")
print(f"Sorunsuz (Arşiv)    : {(df_nlp['problem_var']==0).sum():,}")

print("\nDepartman dağılımı:")
print(df_nlp['departman_tr'].value_counts().to_string())

print("\nOrtalama güven skoru (departmana göre):")
print(
    df_nlp[df_nlp['problem_var']==1]
    .groupby('departman_tr')['guven_skoru']
    .mean().round(3)
    .sort_values(ascending=False)
    .to_string()
)

In [ ]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def ozet_cikar(metin: str, cumle_sayisi: int = 2) -> str:

    metin = str(metin).strip()

    if len(metin) < 100:
        return metin

    try:
        parser     = PlaintextParser.from_string(metin, Tokenizer('english'))
        summarizer = LsaSummarizer()
        ozet       = summarizer(parser.document, cumle_sayisi)
        sonuc      = ' '.join(str(c) for c in ozet)
        return sonuc if sonuc.strip() else metin[:200]
    except Exception:
        return metin[:200]

ornek_metin = df_nlp['review_body'].dropna().iloc[0]
print("Orijinal yorum:")
print(ornek_metin)
print(f"\nÖzet ({len(ornek_metin)} → {len(ozet_cikar(ornek_metin))} karakter):")
print(ozet_cikar(ornek_metin))

In [ ]:
print("Yorumlar özetleniyor...")
tqdm.pandas(desc="Özetleniyor")
df_nlp['ozet'] = df_nlp['review_body'].progress_apply(ozet_cikar)

print(f"Özetleme tamamlandı")
print(f"Ortalama orijinal uzunluk : {df_nlp['body_len'].mean():.0f} karakter")
print(f"Ortalama özet uzunluğu    : {df_nlp['ozet'].str.len().mean():.0f} karakter")

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader_analyzer = SentimentIntensityAnalyzer()

def tam_pipeline(review_body: str, star_rating: int = None) -> dict:

    vader = vader_analyzer.polarity_scores(str(review_body))['compound']

    star_val      = star_rating if star_rating else (1 if vader < -0.3 else 4)
    yildiz_sinyal = int(star_val <= 3)
    vader_sinyal  = int(vader < -0.05)
    problem_skor  = yildiz_sinyal * 0.6 + vader_sinyal * 0.4
    problem_var   = problem_skor >= 0.5

    if problem_var:
        kural_sonuc = kural_ile_departman(review_body)

        if kural_sonuc['guven_skoru'] < 0.45 and 'siniflandirici' in globals():
            try:
                dept_sonuc = zero_shot_departman(review_body)
            except Exception:
                dept_sonuc = kural_sonuc
        else:
            dept_sonuc = kural_sonuc

        departman   = dept_sonuc['departman_tr']
        guven_skoru = dept_sonuc['guven_skoru']
    else:
        departman   = 'Arşiv (Olumlu)'
        guven_skoru = 1.0

    ozet = ozet_cikar(review_body)

    return {
        'ozet':         ozet,
        'problem_var':  problem_var,
        'vader_skoru':  round(vader, 3),
        'problem_skor': round(problem_skor, 2),
        'departman':    departman,
        'guven_skoru':  guven_skoru,
    }

test_yorumlar = [
    ("The screen cracked after one week. Terrible hardware quality.", 1),
    ("App crashes every time I open it. Needs a software fix urgently.", 2),
    ("Package arrived completely destroyed. Box was crushed during shipping.", 1),
    ("Amazing product! Works perfectly and fast delivery. Very happy.", 5),
    ("Customer support never replied to my 5 emails. Unacceptable service.", 2),
]

print("── Tam Pipeline Test Sonuçları ──────────────────────────")
for metin, star in test_yorumlar:
    sonuc = tam_pipeline(metin, star)
    print(f"\nYorum    : {metin[:65]}...")
    print(f"Problem  : {':( Evet' if sonuc['problem_var'] else ':) Hayır'}  "
          f"(skor: {sonuc['problem_skor']})")
    print(f"Departman: {sonuc['departman']}  "
          f"(güven: {sonuc['guven_skoru']:.2f})")
    print(f"Özet     : {sonuc['ozet'][:80]}...")

In [ ]:
CIKTI = DRIVE_PATH + 'faz3_nlp_sonuc.parquet'
df_nlp.to_parquet(CIKTI, index=False, compression='snappy')

print(f"Kaydedildi: {CIKTI}")
print(f"   Satır : {len(df_nlp):,}")
print(f"   Boyut : {os.path.getsize(CIKTI)/1024**2:.0f} MB")
print(f"\nSonraki adım → Faz4_ML_Modelleme.ipynb")